# ⚡ EnerVision AI — Decision Engine & Visualizations

**Standalone Notebook**

This notebook builds the **automated decision & control engine** for the EnerVision AI project, and generates the **key visualizations** used to explain the system's logic and model performance.

| | |
|---|---|
| **Project** | EnerVision AI |
| **Purpose** | Decision logic + reporting visuals |
| **Inputs** | Predicted solar generation & predicted demand |
| **Outputs** | Operational status, recommendations, and 3 chart images |
| **Environment** | Google Colab (CPU is sufficient) |

---


## 📦 Step 1: Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Global plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 300


## 📂 Step 2: Load the Cleaned Dataset

Mounts Google Drive and loads `HomeC_Cleaned.csv` — the same cleaned dataset used to train all three forecasting models. The next two visualizations are computed directly from it.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DATA_PATH = '/content/drive/MyDrive/EnerVision_AI/Data/HomecCleaned.csv'
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"✅ Dataset loaded: {df.shape}")
display(df.head())


## 📈 Step 3: Visualization 1 — Average 24-Hour Diurnal Energy Profile

**Purpose:** highlight the real average mismatch between solar generation and building demand throughout the day — computed by grouping the cleaned dataset by `hour`.


In [ ]:
hourly_profile = df.groupby('hour')[['Solar_kW', 'Demand_kW']].mean().reindex(range(24)).fillna(0)

hours = hourly_profile.index.values
solar_gen = hourly_profile['Solar_kW'].values
demand = hourly_profile['Demand_kW'].values
net_energy = solar_gen - demand

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(hours, solar_gen, label='Solar Generation (kW)', color='#1c59f2', linewidth=2.5, marker='o', markersize=4)
ax.plot(hours, demand, label='Building Demand (kW)', color='#97bbf5', linewidth=2.5, marker='s', markersize=4)
ax.plot(hours, net_energy, label='Net Energy Balance (kW)', color='#3B82F6', linewidth=2, linestyle='--')

ax.axhline(0, color='#6B7280', linewidth=1, linestyle=':')
ax.fill_between(hours, solar_gen, demand, where=(solar_gen >= demand), color='#d2e2fb', alpha=0.15, label='Generation Deficit / Surplus Zone')

ax.set_title('EnerVision AI: Average 24-Hour Diurnal Energy Profile (Cleaned Data)', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Hour of Day (0–23)', fontsize=10)
ax.set_ylabel('Power (kW)', fontsize=10)
ax.set_xticks(range(0, 24, 2))
ax.legend(frameon=True, facecolor='white', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('1_diurnal_energy_profile.png')
plt.show()


## 📊 Step 4: Visualization 2 — Net Energy Thresholds Distribution

**Purpose:** show the real distribution of `Net_Energy` from the cleaned dataset against the same decision-logic boundaries used by the engine (Surplus / Balance / Shortage).


In [ ]:
sample_net_energy = df['Net_Energy'].dropna().values

fig, ax = plt.subplots(figsize=(10, 5))

sns.histplot(sample_net_energy, kde=True, color='#3B82F6', bins=40, stat='density', alpha=0.3, ax=ax)

# Operational Threshold Boundaries (same business rules as the decision engine)
ax.axvline(1.5, color='#10B981', linestyle='--', linewidth=2, label='Surplus Threshold (> +1.5 kW)')
ax.axvline(0.0, color='#EF4444', linestyle='--', linewidth=2, label='Shortage Threshold (< 0.0 kW)')

# Zone boundaries sized to the real data range (with a little padding)
xlim_low = min(sample_net_energy.min(), -1) * 1.05
xlim_high = max(sample_net_energy.max(), 2) * 1.05

ax.axvspan(xlim_low, 0.0, color='#EF4444', alpha=0.08)
ax.axvspan(0.0, 1.5, color='#FBBF24', alpha=0.08)
ax.axvspan(1.5, xlim_high, color='#10B981', alpha=0.08)

ax.set_xlim(xlim_low, xlim_high)

# Annotations, positioned relative to the actual data range
y_max = ax.get_ylim()[1]
ax.text((xlim_low + 0.0) / 2, y_max * 0.85, 'Shortage State 🔴\n(Discharge / Shed)', color='#991B1B', fontweight='bold', ha='center')
ax.text(0.75, y_max * 0.85, 'Balance 🟡\n(Self-Consume)', color='#92400E', fontweight='bold', ha='center')
ax.text((1.5 + xlim_high) / 2, y_max * 0.85, 'Surplus State 🟢\n(Charge / Export)', color='#065F46', fontweight='bold', ha='center')

ax.set_title('Net Energy Differential Distribution & Decision Thresholds (Cleaned Data)', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Net Energy Balance (kW)', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.legend(frameon=True, facecolor='white', fontsize=9, loc='upper right')

plt.tight_layout()
plt.savefig('2_net_energy_thresholds.png')
plt.show()


In [ ]:
df['time'] = pd.to_datetime(df['time'])

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(df['time'], df['Demand_kW'], label='Building Demand (kW)', color='#69a1fa', linewidth=1.5)

ax.set_title('EnerVision AI: Building Demand Over Time', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Time', fontsize=11)
ax.set_ylabel('Building Demand (kW)', fontsize=11)
ax.legend(frameon=True, facecolor='white', fontsize=10)
ax.grid(True, linestyle='--', alpha=0.6)

# Improve x-axis date formatting for readability
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig('4_demand_over_time.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(df['time'], df['Solar_kW'], label='Solar Generation (kW)', color='#1c59f2', linewidth=1.5)
ax.plot(df['time'], df['Demand_kW'], label='Building Demand (kW)', color='#97bbf5', linewidth=1.5)

ax.set_title('EnerVision AI: Solar Generation vs. Building Demand Over Time', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Time', fontsize=11)
ax.set_ylabel('Power (kW)', fontsize=11)
ax.legend(frameon=True, facecolor='white', fontsize=10)
ax.grid(True, linestyle='--', alpha=0.6)

# Improve x-axis date formatting for readability
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig('5_solar_vs_demand.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Select only numerical columns for correlation calculation.
numerical_cols = df.select_dtypes(include=np.number).columns

# Calculate Pearson correlation of all numerical columns with 'Demand_kW'
correlations = df[numerical_cols].corr()['Demand_kW']

# Remove 'Demand_kW' itself from the correlation list as it will be 1.0
correlations = correlations.drop('Demand_kW')

# Sort correlations by absolute value to get the most important features
correlations_abs_sorted = correlations.abs().sort_values(ascending=False)

# Select top N features for better readability, let's say top 10
# If there are fewer than 10 features, it will take all of them.
top_n = 10
features_to_plot = correlations_abs_sorted.head(top_n).index.tolist()

# Re-get the actual correlation values (not absolute) for plotting
actual_correlations = df[features_to_plot + ['Demand_kW']].corr()['Demand_kW'].drop('Demand_kW')

# Ensure the order is maintained as per absolute sorted values
actual_correlations = actual_correlations[features_to_plot]

# Create the bar chart
fig, ax = plt.subplots(figsize=(12, 7))

# Use different colors for positive and negative correlations
colors = ['#10B981' if c > 0 else '#EF4444' for c in actual_correlations.values]

ax.bar(actual_correlations.index, actual_correlations.values, color=colors)

ax.set_title('EnerVision AI: Top Feature Correlations with Building Demand (kW)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Feature', fontsize=11)
ax.set_ylabel('Pearson Correlation Coefficient', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.6)

# Add correlation values on top of bars
for i, v in enumerate(actual_correlations.values):
    ax.text(i, v + (0.02 if v > 0 else -0.05), f'{v:.2f}', color='black', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('6_feature_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

## 📉 Step 5: Visualization 3 — Model Performance Benchmarking

**Purpose:** quantify model accuracy (MAE / RMSE) across the three forecasting models.

The metrics below are **not hardcoded** — they are loaded from `model_results.csv`, a shared results file that each of the three training notebooks (`EnerVision_XGBoost_Model.ipynb`, `EnerVision_BiLSTM_Model.ipynb`, `EnerVision_PatchTSMixer_Model.ipynb`) writes to after evaluating its model. Run those notebooks first (or make sure the file already exists on your Drive) before running this cell.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

RESULTS_PATH = '/content/drive/MyDrive/EnerVision_AI/Models/model_results.csv'

df_results = pd.read_csv(RESULTS_PATH)

# Preferred display order
display_order = ['XGBoost', 'Bi-LSTM', 'PatchTSMixer']

df_results = (
    df_results
    .set_index('model')
    .reindex(display_order)
    .dropna(subset=['mae', 'rmse', 'r2'])
    .reset_index()
)

models = df_results['model'].tolist()
mae = df_results['mae'].tolist()
rmse = df_results['rmse'].tolist()
r2 = df_results['r2'].tolist()

x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))

# MAE
rects1 = ax.bar(
    x - width, mae, width,
    label='MAE',
    color='#9abbf0',
    edgecolor='#14532D'
)

# RMSE
rects2 = ax.bar(
    x, rmse, width,
    label='RMSE',
    color='#4a9eff',
    edgecolor='#14532D'
)

# R²
rects3 = ax.bar(
    x + width, r2, width,
    label='R²',
    color='#5e9fff',
    edgecolor='#14532D'
)

ax.set_title(
    'EnerVision AI: Model Performance Benchmarking',
    fontsize=11,
    fontweight='bold',
    pad=12
)

ax.set_ylabel(
    'Metric Value',
    fontsize=9.5
)

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=9.5)

ax.legend(
    frameon=True,
    facecolor='white',
    fontsize=8.5
)

ax.grid(
    axis='y',
    linestyle='--',
    alpha=0.5
)

# Add exact value labels
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()

        ax.annotate(
            f'{height:.3f}',
            xy=(
                rect.get_x() + rect.get_width() / 2,
                height
            ),
            xytext=(0, 3),
            textcoords="offset points",
            ha='center',
            va='bottom',
            fontsize=8,
            fontweight='bold'
        )

autolabel(rects1)
autolabel(rects2)
autolabel(rects3)

plt.tight_layout()
plt.savefig('3_model_performance.png', dpi=300, bbox_inches='tight')
plt.show()